# Semana 01: Taxonomia dos Sinais Industriais e Arquitetura TI/TA

## Módulo de Sinais Fabris — Fábrica Virtual Smart N1

Este notebook apresenta a **perspectiva de TI e Engenharia de Software** sobre o chão de fábrica, transformando fenômenos físicos do processo produtivo em **sinais elétricos (Discretos, Contínuos e PWM)** e analisando a arquitetura de integração **TI/TA (ISA-95)** como um pipeline de dados e eventos.

### Objetivos de aprendizagem
- Entender a planta física como uma fonte geradora de dados e eventos para sistemas de TI.
- Analisar os 3 tipos fundamentais de sinais industriais: Discreto (0/1), Contínuo (Analógico) e PWM (Modulação por Largura de Pulso).
- Compreender a conversão de grandezas físicas em variáveis de software (Boolean, Float, Integer, Duty Cycle %).
- Mapear os níveis da norma ANSI/ISA-95 sob a ótica de ingestão e transporte de dados.
- Executar scripts Python para decodificação, conversão de ADC de 12 bits e cálculo de sinais PWM.

---


## 1. Fundamentação teórica

### 1.1 A Visão de TI sobre o Chão de Fábrica (OT)

Para um profissional de TI e desenvolvimento de software, a planta industrial não é apenas um conjunto de máquinas mecânicas, mas sim uma **arquitetura distribuída de aquisição de dados e emissão de eventos** em tempo real.

A fusão entre a Tecnologia da Automação (TA) e a Tecnologia da Informação (TI) exige transformar variações de pressão, temperatura, presença física e velocidade em tipos de dados computacionais padronizados.


### 1.2 Os 3 Tipos Fundamentais de Sinais Industriais

![Taxonomia dos Sinais Industriais sob a ótica de TI](img/sinais_industriais_ti.jpg)

Na automação industrial, a comunicação física entre sensores, atuadores e controladores é categorizada em 3 tipos de sinais:

#### 1. Sinal Discreto (Digital Binário: 0/1)
- **Fenômeno Físico:** Presença de peça, contato mecânico acionado, nível máximo atingido.
- **Representação Elétrica:** Tensão contínua $0\text{V DC}$ (Ausência de Sinal / Falso) ou $+24\text{V DC}$ (Sinal Presente / Verdadeiro).
- **Tipo de Dado em Software:** `bool` (`True` / `False`).
- **Exemplo:** Sensor indutivo detectando uma caixa metálica na esteira.

#### 2. Sinal Contínuo (Analógico: 0-10V / 4-20mA)
- **Fenômeno Físico:** Variação progressiva de temperatura ($0$ a $100^\circ\text{C}$), pressão ($0$ a $10\text{ bar}$), vazão de fluido.
- **Representação Elétrica:** Sinal proporcional em tensão ($0\text{ a }10\text{V}$) ou corrente ($4\text{ a }20\text{mA}$).
- **Processamento em TI:** Amostragem por um **Conversor Analógico-Digital (ADC)** de $12\text{ bits}$ ($0\text{ a }4095$) ou $16\text{ bits}$ ($0\text{ a }65535$).
- **Tipo de Dado em Software:** `float` (ex: `temperatura_celsius = 42.8`).

#### 3. Sinal PWM (Pulse Width Modulation — Modulação por Largura de Pulso)
- **Conceito:** Sinal digital quadrado de frequência fixa onde a **largura do pulso ativo ($t_{on}$)** varia em relação ao período total ($T$).
- **Parâmetro Chave — Duty Cycle ($D$):**

$$D = \frac{t_{on}}{T} \times 100\%$$

- **Tensão Média Equivalente ($V_{avg}$):**

$$V_{avg} = D \times V_{max}$$

- **Para que serve o PWM em TI e Automação?**
  1. *Controle de Potência Eficiente:* Permite controlar a velocidade de motores DC/inversores, brilho de LEDs e aquecimento de resistências sem dissipar energia em resistores lineares.
  2. *Emulação de Saída Analógica:* Permite que microcontroladores e CLPs que possuem apenas saídas digitais gerem uma tensão média contínua equivalente sem a necessidade de um conversor DAC (*Digital-to-Analog Converter*) caro.
- **Tipo de Dado em Software:** Percentual `float` ($0.0\%$ a $100.0\%$).


### 1.3 Mapeamento dos Níveis da Norma ISA-95 sob a Ótica de TI

| Nível ISA-95 | Camada Física / Automação | Perspectiva de TI e Software |
| :--- | :--- | :--- |
| **Nível 0** | Sensores, Válvulas, Motores | Fontes primárias de conversão físico-elétrica (0/1, 4-20mA, PWM) |
| **Nível 1** | CLPs, PACs, Cartões de I/O | Amostragem ADC, temporizadores, conversores de sinal e ciclo de varredura |
| **Nível 2** | SCADA, IHMs, Historian | Agregação local, dashboards de observabilidade em tempo real e logs de eventos |
| **Nível 3** | MES / MOM | Gestão de fluxo de produção, controle de lotes e envio de ordens de serviço |
| **Nível 4** | ERP (SAP, TOTVS) | Gestão corporativa, faturamento e integração com pipelines de nuvem |

---


## 2. Arquitetura da atividade

O fluxo de dados da atividade simula o pipeline de aquisição e interpretação de sinais:

**Sinal Físico (Discreto / ADC Analógico 12-bits / PWM Duty Cycle) → Decodificador de TI em Software → Payload JSON de Telemetria**

---


## 3. Prática — Interpretador de Sinais Fabris e Conversores em Python

### Passo 1 — Simular a Decodificação dos 3 Tipos de Sinais
Execute o script abaixo para simular como o software de TI interpreta sinais discretos, converte leituras analógicas de ADC 12-bits e calcula a tensão média equivalente de um sinal PWM.

In [ ]:
import json
import time
import random

def ler_sinal_discreto(tensao_volts):
    # Tensao > 18V representa Nivel Logico ALTO (1/True) em sistemas 24V DC
    estado_logico = tensao_volts >= 18.0
    return {"tipo": "DISCRETO", "tensao_v": tensao_volts, "valor_logico": estado_logico}

def converter_sinal_analogico_adc(valor_adc_12bit, min_eng=0.0, max_eng=10.0):
    # ADC de 12 bits varia de 0 a 4095 (2^12 - 1)
    percentual = valor_adc_12bit / 4095.0
    valor_engenharia = min_eng + (percentual * (max_eng - min_eng))
    return {"tipo": "ANALÓGICO_ADC", "raw_adc": valor_adc_12bit, "valor_medido": round(valor_engenharia, 2)}

def decodificar_sinal_pwm(tempo_ativo_ms, periodo_total_ms, tensao_max_v=24.0):
    duty_cycle_pct = (tempo_ativo_ms / periodo_total_ms) * 100.0
    tensao_media_v = (duty_cycle_pct / 100.0) * tensao_max_v
    return {"tipo": "PWM", "duty_cycle_pct": round(duty_cycle_pct, 1), "tensao_media_v": round(tensao_media_v, 2)}

# Exemplo de amostragem no software
s_discreto = ler_sinal_discreto(24.0)
s_analogico = converter_sinal_analogico_adc(3072, min_eng=0.0, max_eng=100.0) # ex: 0 a 100 °C
s_pwm = decodificar_sinal_pwm(7.5, 10.0, tensao_max_v=24.0) # 75% duty cycle

print("=== SIMULAÇÃO DE DECODIFICAÇÃO DE SINAIS PELO SOFTWARE DE TI ===\n")
print("1. Sinal Discreto: ", s_discreto)
print("2. Sinal Analógico:", s_analogico)
print("3. Sinal PWM:      ", s_pwm)


### Passo 2 — Empacotar as Leituras em Payload JSON de Telemetria
Rode a célula para gerar o payload unificado consumido por APIs REST e brokers MQTT.

In [ ]:
payload_ti = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "estacao_id": "SMART_N1_CELULA_01",
    "sensores": {
        "presenca_peca_bool": s_discreto["valor_logico"],
        "temperatura_processo_celsius": s_analogico["valor_medido"],
        "comando_velocidade_pwm_pct": s_pwm["duty_cycle_pct"],
        "tensao_media_motor_volts": s_pwm["tensao_media_v"]
    }
}

print("=== PAYLOAD JSON UNIFICADO PARA SISTEMAS DE TI ===")
print(json.dumps(payload_ti, indent=2))


---

## 4. Exercícios de fixação e avaliação

### Questão 1
Explique a diferença técnica entre um sinal discreto (binário) e um sinal contínuo (analógico), indicando o tipo de dado utilizado no software de TI para armazenar cada um.

### Questão 2
O que é o Duty Cycle de um sinal PWM e por que a técnica PWM é amplamente utilizada em TI/sistemas embarcados para controlar motores e atuar como um emissor analógico sintético?

### Questão 3
Um conversor ADC de $12\text{ bits}$ de um controlador recebe um sinal de pressão de $0\text{ a }10\text{ bar}$. Se a leitura bruta do registrador for igual a $2048$, qual é a pressão medida traduzida pelo software?
